In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from pytorch_tabnet.tab_model import TabNetClassifier

# =========================
# 读取数据
# =========================
normal = pd.read_csv("./normal.csv")
failure_raw = pd.read_csv("./failure.csv")
failure = failure_raw[failure_raw["model"] == "ST12000NM0008"]
df_data = pd.concat([normal, failure], ignore_index=True)

# =========================
# 特征构造
# =========================
# features = [1,2,3,4,5,7,8,9,10,11,12,13,15,16,17,18,22,23,24,168,170,173,174,175,177,179,
#             180,181,182,183,184,187,188,189,190,191,192,193,194,195,196,197,198,199,200,
#             201,202,206,210,218,220,222,223,224,225,226,231,232,233,234,235,240,241,242,
#             245,247,248,250,251,252,254,255,27,71,82,90,160,161,163,164,165,166,167,169,
#             171,172,176,178,230,244,246,211,212]
features = [5, 9, 187, 188, 193, 194, 197, 198, 241, 242]
features_specified = [f"smart_{f}_raw" for f in features]

X_data = df_data[features_specified]
Y_data = df_data['failure']

# =========================
# 缺失值处理
# =========================
X_data = X_data.fillna(0)

# TabNet 要 numpy
X = X_data.values
y = Y_data.values

print("normal:", len(y) - np.sum(y))
print("failure:", np.sum(y))

# =========================
# 划分数据
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

normal: 2328
failure: 2328


In [ ]:
# 定义 TabNet
clf = TabNetClassifier(
    n_d=16, n_a=16,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_params=dict(lr=2e-2),
    verbose=1
)

# 训练
clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_name=["test"],
    eval_metric=["accuracy"],
    max_epochs=20,
    patience=10,
    batch_size=128,
    virtual_batch_size=64
)
# 预测
y_pred = clf.predict(X_test)

In [ ]:
# =========================
# 评估
# =========================
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"""
TabNet 分类结果:
acc: {acc}
prec: {prec}
recall: {rec}
f1: {f1}
""")

# =========================
# 混淆矩阵
# =========================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Valid', 'Failed'],
            yticklabels=['Valid', 'Failed'])
plt.title('Confusion Matrix - TabNet')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()